# 📊 Exploratory Data Analysis & Modeling — Household Income Prediction

**Dataset:** Regression Dataset for Household Income Analysis  
**Source:** [Kaggle - Stealth Technologies](https://www.kaggle.com/datasets/stealthtechnologies/regression-dataset-for-household-income-analysis)  
**Objective:** Phân tích khám phá dữ liệu (EDA), tiền xử lý, huấn luyện và đánh giá mô hình Random Forest Regression.

### Nội dung phân tích

**Phần I — Exploratory Data Analysis**
1. Import Libraries & Load Data
2. Data Overview & Quality Check
3. Phân phối thu nhập (Income Distribution)
4. Phân tích theo trình độ học vấn (Box Plot)
5. Phân tích theo nghề nghiệp (Grouped Bar)
6. Phân tích theo khu vực (Bar + Pie)
7. Phân tích tương quan (Correlation Heatmap)
8. Phân tích tình trạng việc làm & giới tính
9. Phân tích tình trạng hôn nhân & loại nhà ở
10. Phân tích tuổi & kinh nghiệm (Scatter)
11. Phân phối phần trăm thu nhập (Area Chart)
12. Phân phối trình độ & nghề nghiệp (Doughnut)

**Phần II — Data Preprocessing & Modeling**
13. Data Preprocessing (Log Transform, Label Encoding, Feature Engineering)
14. Train-Test Split
15. Feature Scaling
16. Model Training (5 mô hình hồi quy)
17. Model Evaluation (So sánh, Feature Importance, Learning Curve)
18. Tổng kết & Khuyến nghị

---

# PHẦN I — EXPLORATORY DATA ANALYSIS

---

## 1. Import Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Unified color palette (consistent with Word report) ──────────────────
COLORS = {
    'primary':   '#2E86AB',
    'secondary': '#A23B72',
    'accent':    '#F18F01',
    'success':   '#2ECC71',
    'danger':    '#E74C3C',
    'info':      '#3498DB',
    'dark':      '#2C3E50',
    'palette':   ['#2E86AB', '#A23B72', '#F18F01', '#2ECC71', '#E74C3C',
                  '#9B59B6', '#1ABC9C', '#F39C12', '#3498DB', '#E67E22',
                  '#16A085', '#8E44AD', '#2980B9', '#D35400', '#27AE60'],
}

# Style settings
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'figure.titlesize': 14,
})

print('Libraries imported successfully!')
print(f'Color palette loaded: {len(COLORS["palette"])} colors')

In [ ]:
# Load dataset
df = pd.read_csv('../data/data.csv')
print(f'Dataset shape: {df.shape}')
print(f'Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}')
df.head(10)

## 2. Data Overview & Quality Check

In [ ]:
# Data types and info
print('=== Data Types ===')
print(df.dtypes)
print(f'\n=== Memory Usage ===')
print(f'Total: {df.memory_usage(deep=True).sum() / 1024:.1f} KB')

In [ ]:
# Missing values check
missing = df.isnull().sum()
print('=== Missing Values ===')
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values found!')
print(f'\nTotal missing: {missing.sum()}')
print(f'Duplicate rows: {df.duplicated().sum()}')

In [ ]:
# Statistical summary for numerical columns
df.describe().round(2)

In [ ]:
# Identify column types
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
num_cols = df.select_dtypes(include=['number']).columns.tolist()

print(f'Categorical columns ({len(cat_cols)}):')
for col in cat_cols:
    print(f'  - {col}: {df[col].nunique()} unique values → {df[col].unique().tolist()}')

print(f'\nNumerical columns ({len(num_cols)}):')
for col in num_cols:
    print(f'  - {col}: range [{df[col].min()}, {df[col].max()}]')

## 3. Phân phối thu nhập (Income Distribution)

So sánh phân phối thu nhập gốc với phân phối sau biến đổi log — nhằm xác định chiến lược xử lý skewness cho mô hình.

In [ ]:
log_income = np.log1p(df['Income'])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ── Raw income histogram ──
axes[0].hist(df['Income'], bins=40, color=COLORS['primary'], alpha=0.85,
             edgecolor='white', linewidth=0.5)
axes[0].set_title('Income Distribution (Original)', fontweight='bold')
axes[0].set_xlabel('Annual Income (USD)')
axes[0].set_ylabel('Frequency')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
axes[0].axvline(df['Income'].mean(), color=COLORS['danger'], linestyle='--', linewidth=1.5,
                label=f"Mean: ${df['Income'].mean():,.0f}")
axes[0].axvline(df['Income'].median(), color=COLORS['accent'], linestyle='--', linewidth=1.5,
                label=f"Median: ${df['Income'].median():,.0f}")
axes[0].legend(fontsize=7)

# ── Log income histogram ──
axes[1].hist(log_income, bins=40, color=COLORS['secondary'], alpha=0.85,
             edgecolor='white', linewidth=0.5)
axes[1].set_title('Log(Income) Distribution (After Transform)', fontweight='bold')
axes[1].set_xlabel('log(1 + Income)')
axes[1].set_ylabel('Frequency')
axes[1].axvline(log_income.mean(), color=COLORS['danger'], linestyle='--', linewidth=1.5,
                label=f'Mean: {log_income.mean():.2f}')
axes[1].axvline(log_income.median(), color=COLORS['accent'], linestyle='--', linewidth=1.5,
                label=f'Median: {log_income.median():.2f}')
axes[1].legend(fontsize=7)

plt.tight_layout()
plt.show()

print(f'Skewness (raw):  {df["Income"].skew():.4f}')
print(f'Skewness (log):  {log_income.skew():.4f}')
print(f'Kurtosis (raw):  {df["Income"].kurtosis():.4f}')
print(f'Kurtosis (log):  {log_income.kurtosis():.4f}')

In [ ]:
# Percentile analysis & IQR outlier detection
percentiles_list = [5, 10, 25, 50, 75, 90, 95, 99]
print('=== Income Percentiles ===')
for p in percentiles_list:
    val = df['Income'].quantile(p / 100)
    print(f'  P{p:2d}: ${val:>12,.0f}')

# IQR outlier detection
Q1, Q3 = df['Income'].quantile(0.25), df['Income'].quantile(0.75)
IQR = Q3 - Q1
outliers = df[(df['Income'] < Q1 - 1.5 * IQR) | (df['Income'] > Q3 + 1.5 * IQR)]
print(f'\n=== Outlier Analysis (IQR) ===')
print(f'  Q1: ${Q1:,.0f}  |  Q3: ${Q3:,.0f}  |  IQR: ${IQR:,.0f}')
print(f'  Lower bound: ${Q1 - 1.5 * IQR:,.0f}')
print(f'  Upper bound: ${Q3 + 1.5 * IQR:,.0f}')
print(f'  Outliers: {len(outliers)} ({len(outliers)/len(df)*100:.1f}%)')

## 4. Phân tích theo trình độ học vấn (Box Plot)

Box plot thể hiện phân phối thu nhập theo từng cấp độ học vấn, cho thấy trung vị, IQR, và outliers.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
order = ['High School', "Bachelor's", "Master's", 'Doctorate']
palette_edu = [COLORS['primary'], COLORS['secondary'], COLORS['accent'], COLORS['success']]

sns.boxplot(data=df, x='Education_Level', y='Income', order=order, palette=palette_edu, ax=ax,
            flierprops=dict(marker='o', markersize=2, alpha=0.3))
ax.set_title('Income Distribution by Education Level', fontweight='bold')
ax.set_xlabel('Education Level')
ax.set_ylabel('Annual Income (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))

plt.tight_layout()
plt.show()

# Summary table
print('=== Income by Education Level ===')
edu_stats = df.groupby('Education_Level')['Income'].agg(['count', 'mean', 'median', 'std'])
for level in order:
    if level in edu_stats.index:
        row = edu_stats.loc[level]
        print(f'  {level:15s}  Count: {int(row["count"]):>5,}  '
              f'Mean: ${row["mean"]:>12,.0f}  Median: ${row["median"]:>10,.0f}  '
              f'Std: ${row["std"]:>12,.0f}')

## 5. Phân tích theo nghề nghiệp (Grouped Bar)

So sánh Mean vs Median thu nhập theo nhóm nghề nghiệp — chênh lệch lớn phản ánh phân phối lệch phải.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
occ_stats = df.groupby('Occupation')['Income'].agg(['mean', 'median']).sort_values('median', ascending=False)
x = np.arange(len(occ_stats))
width = 0.35

bars1 = ax.bar(x - width/2, occ_stats['median'], width, label='Median', color=COLORS['primary'], alpha=0.9)
bars2 = ax.bar(x + width/2, occ_stats['mean'], width, label='Mean', color=COLORS['accent'], alpha=0.9)

ax.set_title('Income by Occupation (Mean vs Median)', fontweight='bold')
ax.set_xlabel('Occupation')
ax.set_ylabel('Annual Income (USD)')
ax.set_xticks(x)
ax.set_xticklabels(occ_stats.index, rotation=15, ha='right')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
ax.legend()

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
            f'${bar.get_height()/1000:.0f}K', ha='center', va='bottom', fontsize=7)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
            f'${bar.get_height()/1000:.0f}K', ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.show()

## 6. Phân tích theo khu vực (Bar + Pie)

Thu nhập theo khu vực (trái) kết hợp với phân phối số lượng mẫu theo khu vực (phải).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ── Bar chart: Income by Location ──
loc_stats = df.groupby('Location')['Income'].agg(['mean', 'median']).sort_values('median', ascending=False)
x = np.arange(len(loc_stats))
width = 0.35
axes[0].bar(x - width/2, loc_stats['median'], width, label='Median', color=COLORS['info'], alpha=0.9)
axes[0].bar(x + width/2, loc_stats['mean'], width, label='Mean', color=COLORS['accent'], alpha=0.9)
axes[0].set_title('Income by Location Type', fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(loc_stats.index)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
axes[0].legend()

# ── Pie chart: Location Distribution ──
loc_counts = df['Location'].value_counts()
axes[1].pie(loc_counts, labels=loc_counts.index, autopct='%1.1f%%',
            colors=[COLORS['primary'], COLORS['secondary'], COLORS['accent']],
            startangle=90, textprops={'fontsize': 9})
axes[1].set_title('Location Distribution', fontweight='bold')

plt.tight_layout()
plt.show()

## 7. Phân tích tương quan (Correlation Heatmap)

Ma trận tương quan giữa các biến số — tương quan yếu với Income cho thấy mối quan hệ phi tuyến.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
corr_cols = ['Age', 'Number_of_Dependents', 'Work_Experience', 'Household_Size', 'Income']
corr = df[corr_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

sns.heatmap(corr, mask=mask, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, ax=ax, linewidths=0.5,
            cbar_kws={'label': 'Correlation Coefficient'})
ax.set_title('Correlation Matrix — Numerical Features', fontweight='bold')

plt.tight_layout()
plt.show()

# Print correlation with Income
print('=== Correlation with Income ===')
income_corr = corr['Income'].drop('Income').sort_values(ascending=False)
for feat, c in income_corr.items():
    print(f'  {feat:30s} {c:+.4f}')

In [ ]:
# Correlation bar chart (with all encoded features)
from sklearn.preprocessing import LabelEncoder

df_encoded = df.copy()
for col in cat_cols:
    df_encoded[col] = LabelEncoder().fit_transform(df_encoded[col])

all_corr = df_encoded.corr()['Income'].drop('Income').sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors_corr = [COLORS['success'] if v > 0 else COLORS['danger'] for v in all_corr.values]
all_corr.plot(kind='barh', ax=ax, color=colors_corr, edgecolor='white', alpha=0.8)
ax.set_title('Correlation with Income (All Features)', fontweight='bold', fontsize=14)
ax.set_xlabel('Pearson Correlation Coefficient')
ax.axvline(0, color='gray', linestyle='-', linewidth=0.5)

plt.tight_layout()
plt.show()

print('=== Full Correlation with Income ===')
for feat, c in all_corr.items():
    print(f'  {feat:40s} {c:+.4f}')

## 8. Phân tích tình trạng việc làm & giới tính

So sánh thu nhập Mean vs Median theo Employment Status và Gender.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
width = 0.35

# ── Employment Status ──
emp_stats = df.groupby('Employment_Status')['Income'].agg(['mean', 'median']).sort_values('mean', ascending=False)
x = np.arange(len(emp_stats))
axes[0].bar(x - width/2, emp_stats['median'], width, label='Median', color=COLORS['info'])
axes[0].bar(x + width/2, emp_stats['mean'], width, label='Mean', color=COLORS['danger'])
axes[0].set_title('Income by Employment Status', fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(emp_stats.index, fontsize=9)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
axes[0].legend()

# ── Gender ──
gender_stats = df.groupby('Gender')['Income'].agg(['mean', 'median'])
x = np.arange(len(gender_stats))
axes[1].bar(x - width/2, gender_stats['median'], width, label='Median', color=COLORS['secondary'])
axes[1].bar(x + width/2, gender_stats['mean'], width, label='Mean', color=COLORS['accent'])
axes[1].set_title('Income by Gender', fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(gender_stats.index, fontsize=9)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
axes[1].legend()

plt.tight_layout()
plt.show()

## 9. Phân tích tình trạng hôn nhân & loại nhà ở

Thu nhập theo Marital Status và Housing Type — phát hiện xu hướng nhân khẩu học.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
width = 0.35

# ── Marital Status ──
mar_stats = df.groupby('Marital_Status')['Income'].agg(['mean', 'median']).sort_values('mean', ascending=False)
x = np.arange(len(mar_stats))
axes[0].bar(x - width/2, mar_stats['median'], width, label='Median', color=COLORS['primary'])
axes[0].bar(x + width/2, mar_stats['mean'], width, label='Mean', color=COLORS['accent'])
axes[0].set_title('Income by Marital Status', fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(mar_stats.index, fontsize=9)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
axes[0].legend()

# ── Housing Type ──
house_stats = df.groupby('Type_of_Housing')['Income'].agg(['mean', 'median']).sort_values('mean', ascending=False)
x = np.arange(len(house_stats))
axes[1].bar(x - width/2, house_stats['median'], width, label='Median', color=COLORS['success'])
axes[1].bar(x + width/2, house_stats['mean'], width, label='Mean', color=COLORS['danger'])
axes[1].set_title('Income by Housing Type', fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(house_stats.index, fontsize=8, rotation=10, ha='right')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
axes[1].legend()

plt.tight_layout()
plt.show()

## 10. Phân tích tuổi & kinh nghiệm làm việc (Scatter Plot)

Biểu đồ phân tán với màu sắc theo log(Income) — khám phá mối quan hệ đa chiều.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(df['Work_Experience'], df['Age'], c=np.log1p(df['Income']),
                     cmap='viridis', alpha=0.4, s=8, edgecolors='none')
cbar = plt.colorbar(scatter, ax=ax, label='log(Income)')
ax.set_xlabel('Work Experience (years)')
ax.set_ylabel('Age')
ax.set_title('Age vs Work Experience (colored by log Income)', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Additional: Numerical features vs Income scatter plots
num_features = [c for c in num_cols if c != 'Income']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
scatter_colors = [COLORS['primary'], COLORS['secondary'], COLORS['success'], COLORS['accent']]

for i, col in enumerate(num_features):
    ax = axes[i // 2, i % 2]
    ax.scatter(df[col], df['Income'], alpha=0.15, s=8, color=scatter_colors[i])
    ax.set_title(f'{col} vs Income', fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Income (USD)')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))

    # Add trend line
    z = np.polyfit(df[col], df['Income'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(df[col].min(), df[col].max(), 100)
    ax.plot(x_line, p(x_line), 'r--', alpha=0.7, linewidth=2)

plt.tight_layout()
plt.suptitle('Numerical Features vs Income', y=1.02, fontsize=16, fontweight='bold')
plt.show()

## 11. Phân phối phần trăm thu nhập (Area Chart)

Đường cong percentile P5→P95 cho thấy sự chênh lệch thu nhập rất lớn ở các phần trăm cao.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
percentiles = list(range(5, 100, 5))
values = [df['Income'].quantile(p/100) for p in percentiles]

ax.fill_between(percentiles, values, alpha=0.3, color=COLORS['primary'])
ax.plot(percentiles, values, color=COLORS['primary'], linewidth=2, marker='o', markersize=5)

for i, (p, v) in enumerate(zip(percentiles, values)):
    if p in [25, 50, 75, 90, 95]:
        ax.annotate(f'P{p}: ${v:,.0f}', (p, v), textcoords='offset points',
                    xytext=(0, 12), ha='center', fontsize=7, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                              edgecolor=COLORS['primary'], alpha=0.8))

ax.set_xlabel('Percentile')
ax.set_ylabel('Income (USD)')
ax.set_title('Income Percentile Distribution', fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 12. Phân phối trình độ học vấn & nghề nghiệp (Doughnut Charts)

Biểu đồ donut thể hiện tỷ lệ phân phối các nhóm chính trong tập dữ liệu.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ── Education Level Doughnut ──
edu_counts = df['Education_Level'].value_counts()
wedges1, texts1, autotexts1 = axes[0].pie(
    edu_counts, labels=edu_counts.index, autopct='%1.1f%%',
    colors=COLORS['palette'][:len(edu_counts)], startangle=90,
    pctdistance=0.8, textprops={'fontsize': 9})
centre_circle1 = plt.Circle((0, 0), 0.55, fc='white')
axes[0].add_artist(centre_circle1)
axes[0].set_title('Education Level Distribution', fontweight='bold')
axes[0].text(0, 0, f'N={len(df):,}', ha='center', va='center', fontsize=10, fontweight='bold')

# ── Occupation Doughnut ──
occ_counts = df['Occupation'].value_counts()
wedges2, texts2, autotexts2 = axes[1].pie(
    occ_counts, labels=occ_counts.index, autopct='%1.1f%%',
    colors=COLORS['palette'][:len(occ_counts)], startangle=90,
    pctdistance=0.8, textprops={'fontsize': 9})
centre_circle2 = plt.Circle((0, 0), 0.55, fc='white')
axes[1].add_artist(centre_circle2)
axes[1].set_title('Occupation Distribution', fontweight='bold')
axes[1].text(0, 0, f'N={len(df):,}', ha='center', va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Additional doughnut charts for other categorical features
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
other_cats = ['Employment_Status', 'Gender', 'Marital_Status',
              'Homeownership_Status', 'Type_of_Housing', 'Location']

for i, col in enumerate(other_cats):
    ax = axes[i // 3, i % 3]
    counts = df[col].value_counts()
    wedges, texts, autotexts = ax.pie(
        counts, labels=counts.index, autopct='%1.1f%%',
        colors=COLORS['palette'][:len(counts)], startangle=90,
        pctdistance=0.8, textprops={'fontsize': 8})
    centre = plt.Circle((0, 0), 0.55, fc='white')
    ax.add_artist(centre)
    ax.set_title(col.replace('_', ' '), fontweight='bold', fontsize=10)
    ax.text(0, 0, f'{len(counts)}\ncats', ha='center', va='center', fontsize=8)

plt.suptitle('Categorical Features Distribution', y=1.02, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

# PHẦN II — DATA PREPROCESSING & MODELING

---

## 13. Data Preprocessing

Tiền xử lý dữ liệu gồm 3 bước chính:
- **13.1** Log Transform biến mục tiêu (giảm skewness)
- **13.2** Label Encoding biến phân loại
- **13.3** Feature Engineering (tạo đặc trưng mới)

### 13.1 Log Transform biến mục tiêu

Áp dụng `log1p(Income)` để giảm skewness từ ~2.98 xuống ~1.35, giúp các mô hình hồi quy hoạt động hiệu quả hơn.

In [ ]:
# ── Log Transform ──
df_clean = df.copy()
df_clean['Income_log'] = np.log1p(df_clean['Income'])

print('=== Log Transform ===')
print(f'  Skewness trước:  {df["Income"].skew():.4f}')
print(f'  Skewness sau:    {df_clean["Income_log"].skew():.4f}')
print(f'  Kurtosis trước:  {df["Income"].kurtosis():.4f}')
print(f'  Kurtosis sau:    {df_clean["Income_log"].kurtosis():.4f}')
print(f'\n  ✅ Skewness giảm {abs(df["Income"].skew()) - abs(df_clean["Income_log"].skew()):.2f} đơn vị')

### 13.2 Label Encoding biến phân loại

Chuyển đổi 9 biến phân loại (text) thành dạng số nguyên. Phù hợp với Random Forest vì cây quyết định xử lý tốt giá trị ordinal.

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

# ── Label Encoding ──
label_encoders = {}
encoding_maps = {}

for col in cat_cols:
    le = LabelEncoder()
    df_clean[col + '_encoded'] = le.fit_transform(df_clean[col])
    label_encoders[col] = le
    encoding_maps[col] = dict(zip(le.classes_.tolist(), le.transform(le.classes_).tolist()))

print('=== Label Encoding Results ===')
for col in cat_cols:
    print(f'  {col}:')
    print(f'    {encoding_maps[col]}')

### 13.3 Feature Engineering

Tạo 2 đặc trưng mới nhằm nắm bắt thêm thông tin từ dữ liệu gốc:
- `Age_WorkExp_Ratio`: phản ánh thời điểm bắt đầu sự nghiệp
- `Dependents_per_HHSize`: phản ánh gánh nặng tài chính

In [ ]:
# ── Feature Engineering ──
df_clean['Age_WorkExp_Ratio'] = df_clean['Age'] / (df_clean['Work_Experience'] + 1)
df_clean['Dependents_per_HHSize'] = df_clean['Number_of_Dependents'] / df_clean['Household_Size']

# Distribution of engineered features
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].hist(df_clean['Age_WorkExp_Ratio'], bins=40, color=COLORS['primary'],
             alpha=0.85, edgecolor='white')
axes[0].set_title('Age / Work Experience Ratio', fontweight='bold')
axes[0].set_xlabel('Ratio')
axes[0].set_ylabel('Frequency')

axes[1].hist(df_clean['Dependents_per_HHSize'], bins=20, color=COLORS['secondary'],
             alpha=0.85, edgecolor='white')
axes[1].set_title('Dependents per Household Size', fontweight='bold')
axes[1].set_xlabel('Ratio')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

# Stats & correlation
print('=== Engineered Features Statistics ===')
for feat in ['Age_WorkExp_Ratio', 'Dependents_per_HHSize']:
    corr_val = df_clean[feat].corr(df_clean['Income'])
    print(f'\n  {feat}:')
    print(f'    Mean: {df_clean[feat].mean():.4f}  |  Std: {df_clean[feat].std():.4f}')
    print(f'    Correlation with Income: {corr_val:+.4f}')

In [ ]:
# ── Build final feature list ──
feature_cols = [c for c in num_cols if c != 'Income']
for col in cat_cols:
    feature_cols.append(col + '_encoded')
feature_cols.extend(['Age_WorkExp_Ratio', 'Dependents_per_HHSize'])

X = df_clean[feature_cols]
y_log = df_clean['Income_log']   # Log-transformed target
y_raw = df_clean['Income']       # Original target

print(f'=== Final Feature Set ===')
print(f'  Total features: {len(feature_cols)}')
print(f'  Numerical (original):  {len([c for c in num_cols if c != "Income"])}')
print(f'  Categorical (encoded): {len(cat_cols)}')
print(f'  Engineered:            2')
print(f'\n  Feature columns:')
for i, col in enumerate(feature_cols, 1):
    print(f'    {i:2d}. {col}')
print(f'\n  X shape: {X.shape}')
print(f'  y shape: {y_log.shape}')

## 14. Train-Test Split

Chia dữ liệu thành tập huấn luyện (80%) và tập kiểm tra (20%) với `random_state=42` đảm bảo tái lập kết quả.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train_log, y_test_log = train_test_split(
    X, y_log, test_size=0.2, random_state=42
)

# Convert back to raw scale for evaluation later
y_train_raw = np.expm1(y_train_log)
y_test_raw = np.expm1(y_test_log)

print('=== Train-Test Split ===')
print(f'  Train set: {X_train.shape[0]:,} samples ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'  Test set:  {X_test.shape[0]:,} samples ({X_test.shape[0]/len(X)*100:.0f}%)')
print(f'  Features:  {X_train.shape[1]}')
print(f'\n  y_train_log — Mean: {y_train_log.mean():.4f}  Std: {y_train_log.std():.4f}')
print(f'  y_test_log  — Mean: {y_test_log.mean():.4f}  Std: {y_test_log.std():.4f}')
print(f'\n  y_train_raw — Mean: ${y_train_raw.mean():,.0f}  Median: ${y_train_raw.median():,.0f}')
print(f'  y_test_raw  — Mean: ${y_test_raw.mean():,.0f}  Median: ${np.median(y_test_raw):,.0f}')

## 15. Feature Scaling

Chuẩn hóa đặc trưng bằng `StandardScaler` (mean=0, std=1).  
**Lưu ý:** Random Forest không yêu cầu scaling, nhưng các mô hình tuyến tính (Linear/Ridge/Lasso) cần scaling để hoạt động đúng.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('=== Feature Scaling (StandardScaler) ===')
print(f'  Scaler fitted on {X_train_scaled.shape[0]:,} training samples')
print(f'\n  Before scaling (X_train):')
print(f'    Mean range:  [{X_train.mean().min():.4f}, {X_train.mean().max():.4f}]')
print(f'    Std range:   [{X_train.std().min():.4f}, {X_train.std().max():.4f}]')
print(f'\n  After scaling (X_train_scaled):')
print(f'    Mean range:  [{X_train_scaled.mean(axis=0).min():.6f}, {X_train_scaled.mean(axis=0).max():.6f}]')
print(f'    Std range:   [{X_train_scaled.std(axis=0).min():.4f}, {X_train_scaled.std(axis=0).max():.4f}]')

# Show scaler parameters
print(f'\n  Scaler parameters per feature:')
for i, col in enumerate(feature_cols):
    print(f'    {col:40s}  mean={scaler.mean_[i]:>10.4f}  scale={scaler.scale_[i]:>10.4f}')

## 16. Model Training

Huấn luyện 5 mô hình hồi quy trên biến mục tiêu `log(Income)`:
1. **Linear Regression** — Baseline tuyến tính
2. **Ridge Regression** (α=10.0) — Regularization L2
3. **Lasso Regression** (α=0.01) — Regularization L1
4. **Random Forest** (300 trees, depth=20) — Ensemble cây quyết định
5. **Gradient Boosting** (300 trees, lr=0.05) — Boosting tuần tự

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ── Define models ──
models_config = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression':  Ridge(alpha=10.0),
    'Lasso Regression':  Lasso(alpha=0.01),
    'Random Forest':     RandomForestRegressor(
                             n_estimators=300, max_depth=20,
                             min_samples_split=5, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(
                             n_estimators=300, max_depth=6,
                             learning_rate=0.05, subsample=0.8, random_state=42),
}

print(f'Models to train: {len(models_config)}')
for name, model in models_config.items():
    print(f'  • {name}: {type(model).__name__}')

In [ ]:
# ── Train all models & evaluate ──
results = {}
trained_models = {}

for name, model in models_config.items():
    is_tree = name in ['Random Forest', 'Gradient Boosting']

    # Tree-based models use unscaled features
    if is_tree:
        model.fit(X_train, y_train_log)
        y_pred_log = model.predict(X_test)
        cv_scores = cross_val_score(model, X, y_log, cv=5, scoring='r2', n_jobs=-1)
    else:
        model.fit(X_train_scaled, y_train_log)
        y_pred_log = model.predict(X_test_scaled)
        cv_scores = cross_val_score(model, scaler.fit_transform(X), y_log, cv=5, scoring='r2')

    # Convert predictions back to original scale
    y_pred_raw = np.expm1(y_pred_log)

    results[name] = {
        'MAE':        mean_absolute_error(y_test_raw, y_pred_raw),
        'RMSE':       float(np.sqrt(mean_squared_error(y_test_raw, y_pred_raw))),
        'R2_raw':     r2_score(y_test_raw, y_pred_raw),
        'R2_log':     r2_score(y_test_log, y_pred_log),
        'MAPE':       float(np.mean(np.abs((y_test_raw - y_pred_raw) / y_test_raw)) * 100),
        'CV_R2_Mean': cv_scores.mean(),
        'CV_R2_Std':  cv_scores.std(),
        'y_pred_log': y_pred_log,
        'y_pred_raw': y_pred_raw,
    }
    trained_models[name] = model

    print(f'\n✅ {name}:')
    print(f'   R² (log):  {results[name]["R2_log"]:.4f}')
    print(f'   CV R²:     {results[name]["CV_R2_Mean"]:.4f} ± {results[name]["CV_R2_Std"]:.4f}')
    print(f'   MAE:       ${results[name]["MAE"]:,.0f}')
    print(f'   RMSE:      ${results[name]["RMSE"]:,.0f}')
    print(f'   MAPE:      {results[name]["MAPE"]:.2f}%')

# Identify best model
best_name = max(results, key=lambda k: results[k]['R2_log'])
best_model = trained_models[best_name]
best_results = results[best_name]
print(f'\n{"="*60}')
print(f'🏆 Best Model: {best_name} (R² log = {best_results["R2_log"]:.4f})')
print(f'{"="*60}')

## 17. Model Evaluation

Đánh giá chi tiết hiệu suất mô hình thông qua:
- **17.1** Bảng so sánh tổng hợp
- **17.2** Biểu đồ so sánh R² & CV R²
- **17.3** Biểu đồ so sánh MAE & RMSE
- **17.4** Actual vs Predicted & Residual Distribution
- **17.5** Feature Importance (Random Forest)
- **17.6** Learning Curve

### 17.1 Bảng so sánh tổng hợp

In [ ]:
# ── Comparison table ──
comparison_data = []
for name in results:
    r = results[name]
    best_marker = ' ★' if name == best_name else ''
    comparison_data.append({
        'Model': f'{name}{best_marker}',
        'R² (log)': f'{r["R2_log"]:.4f}',
        'CV R² (5-fold)': f'{r["CV_R2_Mean"]:.4f} ± {r["CV_R2_Std"]:.4f}',
        'MAE': f'${r["MAE"]:,.0f}',
        'RMSE': f'${r["RMSE"]:,.0f}',
        'MAPE': f'{r["MAPE"]:.2f}%',
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df.index = range(1, len(comparison_df) + 1)
comparison_df.index.name = '#'
comparison_df

### 17.2 So sánh R² & Cross-Validation R²

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

model_names = list(results.keys())
r2_log_vals = [results[m]['R2_log'] for m in model_names]
cv_r2_vals = [results[m]['CV_R2_Mean'] for m in model_names]
cv_r2_std = [results[m]['CV_R2_Std'] for m in model_names]
colors_bar = [COLORS['success'] if m == best_name else COLORS['primary'] for m in model_names]

# ── R² (log scale) ──
bars = axes[0].bar(range(len(model_names)), r2_log_vals, color=colors_bar, alpha=0.9, edgecolor='white')
axes[0].set_xticks(range(len(model_names)))
axes[0].set_xticklabels([n.replace(' ', '\n') for n in model_names], fontsize=8)
axes[0].set_ylabel('R² Score (log scale)')
axes[0].set_title('Model Comparison — R² (Log Scale)', fontweight='bold')
for bar, val in zip(bars, r2_log_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.002,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

# ── CV R² with error bars ──
axes[1].bar(range(len(model_names)), cv_r2_vals, yerr=cv_r2_std,
            color=colors_bar, alpha=0.9, edgecolor='white', capsize=5)
axes[1].set_xticks(range(len(model_names)))
axes[1].set_xticklabels([n.replace(' ', '\n') for n in model_names], fontsize=8)
axes[1].set_ylabel('CV R² Score (5-fold)')
axes[1].set_title('Cross-Validation R² (5-Fold)', fontweight='bold')
for i, (val, std) in enumerate(zip(cv_r2_vals, cv_r2_std)):
    axes[1].text(i, val + std + 0.003, f'{val:.4f}\n±{std:.4f}',
                 ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.show()

### 17.3 So sánh MAE & RMSE

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

mae_vals = [results[m]['MAE'] for m in model_names]
rmse_vals = [results[m]['RMSE'] for m in model_names]

# ── MAE ──
bars_mae = axes[0].bar(range(len(model_names)), mae_vals, color=colors_bar, alpha=0.9, edgecolor='white')
axes[0].set_xticks(range(len(model_names)))
axes[0].set_xticklabels([n.replace(' ', '\n') for n in model_names], fontsize=8)
axes[0].set_ylabel('MAE (USD)')
axes[0].set_title('Model Comparison — MAE', fontweight='bold')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.2f}M'))
for bar, val in zip(bars_mae, mae_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height(),
                 f'${val/1000:,.0f}K', ha='center', va='bottom', fontsize=7)

# ── RMSE ──
bars_rmse = axes[1].bar(range(len(model_names)), rmse_vals, color=colors_bar, alpha=0.9, edgecolor='white')
axes[1].set_xticks(range(len(model_names)))
axes[1].set_xticklabels([n.replace(' ', '\n') for n in model_names], fontsize=8)
axes[1].set_ylabel('RMSE (USD)')
axes[1].set_title('Model Comparison — RMSE', fontweight='bold')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.2f}M'))
for bar, val in zip(bars_rmse, rmse_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height(),
                 f'${val/1000:,.0f}K', ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.show()

### 17.4 Actual vs Predicted & Residual Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

y_pred_log = best_results['y_pred_log']

# ── Actual vs Predicted (log scale) ──
axes[0].scatter(y_test_log, y_pred_log, alpha=0.3, s=8, color=COLORS['primary'])
min_val = min(y_test_log.min(), y_pred_log.min())
max_val = max(y_test_log.max(), y_pred_log.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=1.5, label='Perfect Prediction')
axes[0].set_xlabel('Actual log(Income)')
axes[0].set_ylabel('Predicted log(Income)')
axes[0].set_title(f'Actual vs Predicted (Log Scale)\nR² = {best_results["R2_log"]:.4f}', fontweight='bold')
axes[0].legend()

# ── Residual Distribution ──
residuals = y_test_log.values - y_pred_log
axes[1].hist(residuals, bins=40, color=COLORS['secondary'], alpha=0.85, edgecolor='white')
axes[1].axvline(0, color=COLORS['danger'], linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Residual (Actual - Predicted)')
axes[1].set_ylabel('Frequency')
axes[1].set_title(f'Residual Distribution\nMean={residuals.mean():.4f}, Std={residuals.std():.4f}', fontweight='bold')

plt.tight_layout()
plt.show()

### 17.5 Feature Importance (Random Forest)

In [ ]:
# ── Feature Importance ──
fi = pd.Series(best_model.feature_importances_, index=feature_cols).sort_values(ascending=False)

# Table
print(f'=== Feature Importance — {best_name} ===')
for rank, (feat, imp) in enumerate(fi.items(), 1):
    feat_clean = feat.replace('_encoded', '').replace('_', ' ')
    bar = '█' * int(imp * 100)
    print(f'  {rank:2d}. {feat_clean:35s} {imp:.4f} ({imp*100:.2f}%)  {bar}')

In [ ]:
# ── Feature Importance Chart (Horizontal Bar) ──
fig, ax = plt.subplots(figsize=(10, 6))
fi_sorted = fi.sort_values(ascending=True)
colors_fi = plt.cm.viridis(np.linspace(0.2, 0.9, len(fi_sorted)))

bars = ax.barh(range(len(fi_sorted)), fi_sorted.values, color=colors_fi,
               edgecolor='white', linewidth=0.5)
ax.set_yticks(range(len(fi_sorted)))
feature_labels = [f.replace('_encoded', '').replace('_', ' ') for f in fi_sorted.index]
ax.set_yticklabels(feature_labels, fontsize=9)
ax.set_xlabel('Importance Score')
ax.set_title(f'{best_name} — Feature Importance', fontweight='bold')

for i, (val, bar) in enumerate(zip(fi_sorted.values, bars)):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2, f'{val:.2%}',
            va='center', fontsize=8, fontweight='bold')

ax.set_xlim(0, fi_sorted.max() * 1.2)
plt.tight_layout()
plt.show()

### 17.6 Learning Curve

In [ ]:
from sklearn.model_selection import learning_curve

# ── Learning Curve (using lighter RF for speed) ──
rf_lc = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
train_sizes, train_scores, test_scores = learning_curve(
    rf_lc, X, y_log, cv=5, n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10),
    scoring='r2'
)

train_mean = train_scores.mean(axis=1)
train_std = train_scores.std(axis=1)
test_mean = test_scores.mean(axis=1)
test_std = test_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(10, 5))
ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std,
                alpha=0.15, color=COLORS['primary'])
ax.fill_between(train_sizes, test_mean - test_std, test_mean + test_std,
                alpha=0.15, color=COLORS['danger'])
ax.plot(train_sizes, train_mean, 'o-', color=COLORS['primary'], linewidth=2, label='Training Score')
ax.plot(train_sizes, test_mean, 'o-', color=COLORS['danger'], linewidth=2, label='Validation Score')

ax.set_xlabel('Training Set Size')
ax.set_ylabel('R² Score')
ax.set_title('Learning Curve — Random Forest', fontweight='bold')
ax.legend(loc='best')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Training score at max size: {train_mean[-1]:.4f}')
print(f'Validation score at max size: {test_mean[-1]:.4f}')
print(f'Gap (overfitting indicator): {train_mean[-1] - test_mean[-1]:.4f}')

---

## 18. Tổng kết & Khuyến nghị

### Dataset Characteristics
- **10,000 records** with **13 features** (4 numerical + 9 categorical) and 1 target variable (Income)
- **No missing values** and **no duplicate** rows
- Dataset is **synthetic** with added noise

### Income Distribution
- Income is **heavily right-skewed** (skewness ≈ 2.98)
- Mean ($816K) is **~11x higher** than Median ($72K), indicating extreme outliers
- ~19% of data are outliers (IQR method)
- **Log transformation** effectively reduces skewness to ~1.35

### Feature Relationships
- **Weak linear correlations** between features and income (max |r| < 0.2)
- The weak correlations suggest **non-linear relationships** → tree-based models perform better
- Engineered feature **Age/WorkExp Ratio** is the most important predictor (17.08%)

### Model Results
- **Random Forest** is the best model: R² (log) = 0.1472, CV R² = 0.1527 ± 0.011
- Outperforms linear models by **~4.5x** in R² (log)
- **Gradient Boosting** is second best with R² (log) = 0.0471
- Modest R² due to high noise in synthetic data, but results are **statistically consistent** across CV folds

### Top 3 Feature Importance
1. **Age/WorkExp Ratio** (17.08%) — khi nào bắt đầu sự nghiệp
2. **Work Experience** (13.40%) — kinh nghiệm tích lũy
3. **Age** (12.26%) — tuổi tác

### Khuyến nghị cải thiện
1. **Thu thập thêm dữ liệu** — bổ sung biến tài sản, khoản vay, lịch sử thu nhập
2. **Thử One-Hot Encoding** — cho các biến danh nghĩa trong mô hình tuyến tính
3. **Hyperparameter Tuning** — RandomizedSearchCV hoặc Bayesian Optimization
4. **Stacking/Blending** — kết hợp nhiều mô hình
5. **Xử lý Outlier nâng cao** — Winsorization hoặc Robust Scaler
6. **Feature Engineering** — thêm interaction features và polynomial features

In [ ]:
print('=' * 60)
print('  📊 EDA & Modeling Complete!')
print('  Charts are consistent with Word report')
print('  (Bao_cao_Phan_tich_Thu_nhap_Ho_Gia_dinh.docx)')
print()
print(f'  🏆 Best Model: {best_name}')
print(f'     R² (log):     {best_results["R2_log"]:.4f}')
print(f'     CV R² (5-fold): {best_results["CV_R2_Mean"]:.4f}')
print(f'     MAE:          ${best_results["MAE"]:,.0f}')
print()
print('  To run the web app:')
print('    python main.py')
print('=' * 60)